In [1]:
!pip install -q requests beautifulsoup4 langchain-text-splitters langchain-huggingface sentence-transformers langchain-community faiss-cpu langchain-google-genai gradio scikit-learn "langchain-core>=1.8.4,<2.0.0"

In [2]:
import requests
from bs4 import BeautifulSoup

book_url = "https://bn.wikisource.org/wiki/স্পেনীয়_মুসলমান_সভ্যতা"

headers = {
    "User-Agent": "Mozilla/5.0 (Educational Bangla RAG Project)"
}

response = requests.get(book_url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.get_text(strip=True))
print("HTTP Status:", response.status_code)

Page title: স্পেনীয় মুসলমান সভ্যতা - উইকিসংকলন একটি মুক্ত পাঠাগার
HTTP Status: 200


In [3]:
from urllib.parse import urljoin, urlparse, unquote

chapter_links = []

for link in soup.find_all("a", href=True):
    chapter_name = link.get_text(strip=True)
    chapter_url = urljoin(book_url, link["href"])

    if ("অধ্যায়" in chapter_name and
        unquote(urlparse(chapter_url).path).startswith(
            "/wiki/স্পেনীয়_মুসলমান_সভ্যতা/"
        )):

        if not any(ch["url"] == chapter_url for ch in chapter_links):
            chapter_links.append({
                "chapter": chapter_name,
                "url": chapter_url
            })

print("Total chapters found:", len(chapter_links))

for chapter in chapter_links:
    print(chapter["chapter"], ":", chapter["url"])


Total chapters found: 11
প্রথম অধ্যায় : https://bn.wikisource.org/wiki/%E0%A6%B8%E0%A7%8D%E0%A6%AA%E0%A7%87%E0%A6%A8%E0%A7%80%E0%A6%AF%E0%A6%BC_%E0%A6%AE%E0%A7%81%E0%A6%B8%E0%A6%B2%E0%A6%AE%E0%A6%BE%E0%A6%A8_%E0%A6%B8%E0%A6%AD%E0%A7%8D%E0%A6%AF%E0%A6%A4%E0%A6%BE/%E0%A6%AA%E0%A7%8D%E0%A6%B0%E0%A6%A5%E0%A6%AE_%E0%A6%85%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC
দ্বিতীয় অধ্যায় : https://bn.wikisource.org/wiki/%E0%A6%B8%E0%A7%8D%E0%A6%AA%E0%A7%87%E0%A6%A8%E0%A7%80%E0%A6%AF%E0%A6%BC_%E0%A6%AE%E0%A7%81%E0%A6%B8%E0%A6%B2%E0%A6%AE%E0%A6%BE%E0%A6%A8_%E0%A6%B8%E0%A6%AD%E0%A7%8D%E0%A6%AF%E0%A6%A4%E0%A6%BE/%E0%A6%A6%E0%A7%8D%E0%A6%AC%E0%A6%BF%E0%A6%A4%E0%A7%80%E0%A6%AF%E0%A6%BC_%E0%A6%85%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC
তৃতীয় অধ্যায় : https://bn.wikisource.org/wiki/%E0%A6%B8%E0%A7%8D%E0%A6%AA%E0%A7%87%E0%A6%A8%E0%A7%80%E0%A6%AF%E0%A6%BC_%E0%A6%AE%E0%A7%81%E0%A6%B8%E0%A6%B2%E0%A6%AE%E0%A6%BE%E0%A6%A8_%E0%A6%B8%E0%A6%AD%E0%A7%8D%E0%A6%AF%E0%A6%A4%E0%A6%BE/%E0%A6%

In [4]:
import time

chapter_documents = []

for chapter in chapter_links:
    response = requests.get(
        chapter["url"], headers=headers, timeout=30
    )
    response.raise_for_status()

    chapter_soup = BeautifulSoup(response.text, "html.parser")

    content = (
        chapter_soup.select_one(".prp-pages-output")
        or chapter_soup.select_one("#mw-content-text .mw-parser-output")
    )

    if content is None:
        raise ValueError(f"Content not found: {chapter['chapter']}")

    for element in content.select(
        "script, style, .mw-editsection, .noprint, .ws-noexport"
    ):
        element.decompose()

    chapter_text = content.get_text(" ", strip=True)

    chapter_documents.append({
        "chapter": chapter["chapter"],
        "url": chapter["url"],
        "text": chapter_text
    })

    print(f"{chapter['chapter']}: {len(chapter_text)} characters")
    time.sleep(0.5)

print("\nTotal chapters downloaded:", len(chapter_documents))
print("\nFirst chapter preview:")
print(chapter_documents[0]["text"][:500])

প্রথম অধ্যায়: 13943 characters
দ্বিতীয় অধ্যায়: 3417 characters
তৃতীয় অধ্যায়: 8919 characters
চতুর্থ অধ্যায়: 10437 characters
পঞ্চম অধ্যায়: 3691 characters
ষষ্ঠ অধ্যায়: 3446 characters
সপ্তম অধ্যায়: 2325 characters
অষ্টম অধ্যায়: 6582 characters
নবম অধ্যায়: 2162 characters
দশম অধ্যায়: 630 characters
একাদশ অধ্যায়: 2975 characters

Total chapters downloaded: 11

First chapter preview:
﻿ স্পেনীয় মুসলমান সভ্যতা। প্রাচীনকালে সভ্যতা, সৌন্দর্য্য ও শিক্ষার বিচিত্র লীলাভূমি ও কীর্ত্তিমন্দির বলিয়া যে সমস্ত মহানগরী খ্যাতিলাভ করিয়াছিল, তন্মধ্যে গৌরবোন্নত, সৌন্দর্য্য-সমলঙ্কত, সমৃদ্ধিসম্পন্ন স্পেনের কর্ডোভা মহানগরী অন্যতম। বোগদাদ ব্যতীত কর্ডোভা মহানগরীর সহিত অপর কোনও নগরীর নামও উল্লিখিত হইবার যোগ্য নহে। স্পেনকে পরী বলিয়া কল্পনা করিলে কর্ডোভাকে তাহার চক্ষু বলিয়া স্থান দিতে হয়। প্রাচীন আরব ঐতিহাসিকগণ কর্ডোভাকে স্পেনের পাত্রী বা ক’নে (Bride) বলিয়া বর্ণনা করিয়া গিয়াছেন। গৌরবের দিনে 


In [5]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = []

for chapter in chapter_documents:
    clean_text = re.sub(r"\s+", " ", chapter["text"]).strip()
    clean_text = clean_text.replace("\ufeff", "")

    documents.append(
        Document(
            page_content=clean_text,
            metadata={
                "chapter": chapter["chapter"],
                "source": chapter["url"]
            }
        )
    )

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = text_splitter.split_documents(documents)

print("Original chapters:", len(documents))
print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content[:400])
print("\nMetadata:", chunks[0].metadata)

Original chapters: 11
Total chunks: 90

First chunk:
স্পেনীয় মুসলমান সভ্যতা। প্রাচীনকালে সভ্যতা, সৌন্দর্য্য ও শিক্ষার বিচিত্র লীলাভূমি ও কীর্ত্তিমন্দির বলিয়া যে সমস্ত মহানগরী খ্যাতিলাভ করিয়াছিল, তন্মধ্যে গৌরবোন্নত, সৌন্দর্য্য-সমলঙ্কত, সমৃদ্ধিসম্পন্ন স্পেনের কর্ডোভা মহানগরী অন্যতম। বোগদাদ ব্যতীত কর্ডোভা মহানগরীর সহিত অপর কোনও নগরীর নামও উল্লিখিত হইবার যোগ্য নহে। স্পেনকে পরী বলিয়া কল্পনা করিলে কর্ডোভাকে তাহার চক্ষু বলিয়া স্থান দিতে হয়। প্রাচীন আ

Metadata: {'chapter': 'প্রথম অধ্যায়', 'source': 'https://bn.wikisource.org/wiki/%E0%A6%B8%E0%A7%8D%E0%A6%AA%E0%A7%87%E0%A6%A8%E0%A7%80%E0%A6%AF%E0%A6%BC_%E0%A6%AE%E0%A7%81%E0%A6%B8%E0%A6%B2%E0%A6%AE%E0%A6%BE%E0%A6%A8_%E0%A6%B8%E0%A6%AD%E0%A7%8D%E0%A6%AF%E0%A6%A4%E0%A6%BE/%E0%A6%AA%E0%A7%8D%E0%A6%B0%E0%A6%A5%E0%A6%AE_%E0%A6%85%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC'}


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

test_text = "কর্ডোভা নগরীর বৈশিষ্ট্য কী?"

test_vector = embedding_model.embed_query(test_text)

print("Embedding model loaded successfully!")
print("Vector dimension:", len(test_vector))
print("First 5 values:", test_vector[:5])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded successfully!
Vector dimension: 384
First 5 values: [0.13958428800106049, -0.014853318221867085, 0.013031278736889362, 0.004822175484150648, -0.07996479421854019]


In [7]:
from langchain_community.vectorstores import FAISS

# Create the FAISS vector database
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# Create a LangChain retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Test retrieval
question = "কর্ডোভা নগরী কেন বিখ্যাত ছিল?"

retrieved_docs = retriever.invoke(question)

print("Number of retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nResult {i}")
    print("Chapter:", doc.metadata["chapter"])
    print("Text:", doc.page_content[:300])

/tmp/ipykernel_56704/2392325909.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Number of retrieved chunks: 3

Result 1
Chapter: নবম অধ্যায়
Text: হইত। হায়! স্পেন, তোমার সেই গৌরবকাহিনী কে আর স্মরণ কবে? তুমি মোস্‌লেম গৌরবের মহা সমাধি! তোমার অণু রেণুতে মোস্‌লেম মহিমা বিজড়িত!

Result 2
Chapter: প্রথম অধ্যায়
Text: স্পেনীয় মুসলমান সভ্যতা। প্রাচীনকালে সভ্যতা, সৌন্দর্য্য ও শিক্ষার বিচিত্র লীলাভূমি ও কীর্ত্তিমন্দির বলিয়া যে সমস্ত মহানগরী খ্যাতিলাভ করিয়াছিল, তন্মধ্যে গৌরবোন্নত, সৌন্দর্য্য-সমলঙ্কত, সমৃদ্ধিসম্পন্ন স্পেনের কর্ডোভা মহানগরী অন্যতম। বোগদাদ ব্যতীত কর্ডোভা মহানগরীর সহিত অপর কোনও নগরীর নামও উল্লিখিত হইব

Result 3
Chapter: অষ্টম অধ্যায়
Text: সরোবরগুলি প্রমোদিত থাকিত। দরবার প্রাসাদে ৪টী সিংহের উপরে একটা অপূর্ব্ব সিংহাসন রক্ষিত ছিল। সিংহমুর্ত্তিগুলি যার পর নাই রমণীয় ছিল। এই প্রাসাদটী একশত আটাশটী অতি দীর্ঘ এবং চমৎকারগঠন স্তম্ভের উপর অবস্থিত ছিল। ইহার বিরাট কক্ষ সৌন্দর্য্য ও বিশালতায় অতুলনীয় ছিল! স্তম্ভাবলীর শীর্ষদেশে অতি মনোহর লতা কুঞ্জ


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build character-based TF-IDF index
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)

tfidf_matrix = tfidf.fit_transform(
    [doc.page_content for doc in chunks]
)

class HybridRetriever:
    def invoke(self, question):
        # Semantic search using FAISS
        semantic_docs = vectorstore.similarity_search(
            question, k=6
        )

        # Character-based similarity search
        query_vector = tfidf.transform([question])
        scores = cosine_similarity(
            query_vector, tfidf_matrix
        ).flatten()

        top_indices = scores.argsort()[::-1][:6]
        lexical_docs = [chunks[i] for i in top_indices]

        # Combine results without duplicates
        combined = []
        seen = set()

        for doc in lexical_docs[:4] + semantic_docs[:4]:
            key = (doc.metadata["source"], doc.page_content)

            if key not in seen:
                combined.append(doc)
                seen.add(key)

        return combined

retriever = HybridRetriever()

print("Hybrid retriever created successfully!")

Hybrid retriever created successfully!


In [9]:
import os
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = getpass.getpass(
    "Enter your Gemini API key: "
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

response = llm.invoke(
    "বাংলায় একটি বাক্যে নিজের পরিচয় দাও।"
)

print("LLM connected successfully!")
print(response.content)

Enter your Gemini API key: ··········
LLM connected successfully!
[{'type': 'text', 'text': 'আমি একটি কৃত্রিম বুদ্ধিমত্তা (AI), যা বাংলা ভাষায় কথা বলতে এবং আপনার বিভিন্ন প্রশ্নের উত্তর দিতে তৈরি করা হয়েছে।', 'extras': {'signature': 'Eo4BCosBAWkUfROfBfighjCjJI4/oUfn6YHCjurfUnu2KNNe+UNWHlhYTvztibqCtEhANJOUW+G3+ROlpuXzm81I/EOmBFZvCw+UErXZlVgg5ZPEtuHy3ryhLc8zn5S80ZIjAX2KYDfxuUbLqBDrOopjHKG6hxQuMw4EQfEt4WWifGMijw4imWUCeJJzbprpNA=='}}]


In [10]:
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NO_ANSWER = "এই বইয়ে প্রশ্নটির উত্তর পাওয়া যায়নি।"

In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    তুমি 'স্পেনীয় মুসলমান সভ্যতা' বইয়ের একজন প্রশ্নোত্তর সহকারী।

    নিয়ম:
    ১. শুধুমাত্র প্রদত্ত বইয়ের অংশ থেকে উত্তর দেবে।
    ২. সবগুলো প্রাসঙ্গিক অংশ মনোযোগ দিয়ে পড়বে।
    ৩. পুরনো বাংলা বানান, সমার্থক শব্দ এবং বাক্যের
       প্রসঙ্গ বিবেচনা করে উত্তর খুঁজবে।
    ৪. উত্তর স্পষ্টভাবে পাওয়া গেলে সংক্ষিপ্ত ও
       সঠিক উত্তর দেবে।
    ৫. উত্তরের সাথে প্রাসঙ্গিক সূত্র নম্বর
       [1], [2] ইত্যাদি উল্লেখ করবে।
    ৬. কোনো অংশেই উত্তর সমর্থিত না হলে শুধু লিখবে:
       এই বইয়ে প্রশ্নটির উত্তর পাওয়া যায়নি।
    ৭. নিজের জ্ঞান থেকে কোনো তথ্য যোগ করবে না।
    ৮. উত্তর বাংলায় লিখবে।
    """),
    ("human", "বইয়ের অংশ:\n{context}\n\nপ্রশ্ন: {question}")
])

rag_chain = prompt | llm | StrOutputParser()

print("RAG prompt updated successfully!")

RAG prompt updated successfully!


In [12]:
import gradio as gr
import re
from langchain_core.documents import Document

# Detect no-answer responses even when Gemini adds citations
NO_ANSWER_STEM = NO_ANSWER.rstrip("।").strip()

def chatbot_reply(message, history):

    # 1. Retrieve relevant passages
    docs = retriever.invoke(message)

    if not docs:
        return NO_ANSWER

    context = "\n\n".join(
        f"[{i}] অধ্যায়: {doc.metadata['chapter']}\n"
        f"পাঠ্যাংশ: {doc.page_content}"
        for i, doc in enumerate(docs, 1)
    )

    # 2. Generate answer
    answer = rag_chain.invoke({
        "context": context,
        "question": message
    }).strip()

    # 3. Full-chapter fallback
    if NO_ANSWER_STEM in answer:
        top_chapter = docs[0].metadata["chapter"]

        full_chapter = next(
            (
                ch for ch in chapter_documents
                if ch["chapter"] == top_chapter
            ),
            None
        )

        if full_chapter is not None:
            docs = [
                Document(
                    page_content=full_chapter["text"],
                    metadata={
                        "chapter": full_chapter["chapter"],
                        "source": full_chapter["url"]
                    }
                )
            ]

            answer = rag_chain.invoke({
                "context": (
                    f"[1] অধ্যায়: {top_chapter}\n"
                    f"পাঠ্যাংশ: {full_chapter['text']}"
                ),
                "question": message
            }).strip()

    # 4. No-answer handling
    if NO_ANSWER_STEM in answer:
        return NO_ANSWER

    # 5. Recognize [1], [1, 2], [১], [১, ২]
    citation_groups = re.findall(
        r"\[([0-9০-৯,\s]+)\]", answer
    )

    cited = set()

    for group in citation_groups:
        for number in group.split(","):
            number = number.strip()

            if number.isdecimal():
                index = int(number)

                if 1 <= index <= len(docs):
                    cited.add(index)

    if not cited:
        return NO_ANSWER

    # 6. Format unique chapter sources
    sources = []
    seen = set()

    for i in sorted(cited):
        doc = docs[i - 1]
        chapter = doc.metadata["chapter"]
        url = doc.metadata["source"]

        if url not in seen:
            sources.append(f"- [{chapter}]({url})")
            seen.add(url)

    return answer + "\n\n**সূত্র:**\n" + "\n".join(sources)


# 7. Create Gradio interface
demo = gr.ChatInterface(
    fn=chatbot_reply,
    title="স্পেনীয় মুসলমান সভ্যতা - RAG Chatbot",
    description="এই বই সম্পর্কে বাংলায় প্রশ্ন করুন।",
    examples=[
        "কর্ডোভা নগরী কেন বিখ্যাত ছিল?",
        "১৯৬৯ সালে চাঁদে প্রথম কে অবতরণ করেন?"
    ]
)

In [13]:
test_cases = [
    {
        "question": "কর্ডোভা নগরী কেন বিখ্যাত ছিল?",
        "expected": "সভ্যতা, শিক্ষা, সৌন্দর্য, শিল্প ও বাণিজ্যের জন্য",
        "chapter": "প্রথম অধ্যায়"
    },
    {
        "question": "কর্ডোভার জামে মসজিদের পত্তন কে করেন এবং কবে?",
        "expected": "সোলতান প্রথম আব্দর রহমান, ৭৮৪ খ্রিস্টাব্দে",
        "chapter": "দ্বিতীয় অধ্যায়"
    },
    {
        "question": "কর্ডোভার জামে মসজিদের নির্মাণ কে সম্পন্ন করেন?",
        "expected": "সোলতান হিশাম, ৭৯৭ খ্রিস্টাব্দে",
        "chapter": "দ্বিতীয় অধ্যায়"
    },
    {
        "question": "আজ্‌ জোহরা নগরী কে প্রতিষ্ঠা করেন?",
        "expected": "খলিফা তৃতীয় আব্দর রহমান",
        "chapter": "তৃতীয় অধ্যায়"
    },
    {
        "question": "কর্ডোভায় সাধারণের জন্য কতটি বিরাট লাইব্রেরি ছিল?",
        "expected": "১৭টি",
        "chapter": "চতুর্থ অধ্যায়"
    },
    {
        "question": "খলিফা হাকেম কতটি মূল্যবান গ্রন্থ সংগ্রহ করেছিলেন?",
        "expected": "প্রায় ছয় লক্ষ",
        "chapter": "চতুর্থ অধ্যায়"
    },
    {
        "question": "আলহামরা প্রাসাদের ভিত্তি কে স্থাপন করেন?",
        "expected": "সোলতান ইবনে আলআহমর",
        "chapter": "অষ্টম অধ্যায়"
    },
    {
        "question": "গ্রাণাডায় কোন কোন শিল্পের উন্নতি হয়েছিল?",
        "expected": "কাগজ, রেশমী বস্ত্র, অস্ত্রশস্ত্র, খোদাই ও অন্যান্য শিল্প",
        "chapter": "দশম অধ্যায়"
    },
    {
        "question": "গ্রাণাডায় কতটি কলেজ ও উচ্চশ্রেণীর বিদ্যালয় প্রতিষ্ঠিত হয়েছিল?",
        "expected": "২০০টি",
        "chapter": "একাদশ অধ্যায়"
    },
    {
        "question": "১৯৬৯ সালে চাঁদে প্রথম কে অবতরণ করেন?",
        "expected": "এই বইয়ে প্রশ্নটির উত্তর পাওয়া যায়নি।",
        "chapter": "Not applicable"
    }
]

print("Total test questions:", len(test_cases))

for i, test in enumerate(test_cases, 1):
    print(f"{i}. {test['question']}")
    print(f"   Expected: {test['expected']}")
    print(f"   Chapter: {test['chapter']}\n")

Total test questions: 10
1. কর্ডোভা নগরী কেন বিখ্যাত ছিল?
   Expected: সভ্যতা, শিক্ষা, সৌন্দর্য, শিল্প ও বাণিজ্যের জন্য
   Chapter: প্রথম অধ্যায়

2. কর্ডোভার জামে মসজিদের পত্তন কে করেন এবং কবে?
   Expected: সোলতান প্রথম আব্দর রহমান, ৭৮৪ খ্রিস্টাব্দে
   Chapter: দ্বিতীয় অধ্যায়

3. কর্ডোভার জামে মসজিদের নির্মাণ কে সম্পন্ন করেন?
   Expected: সোলতান হিশাম, ৭৯৭ খ্রিস্টাব্দে
   Chapter: দ্বিতীয় অধ্যায়

4. আজ্‌ জোহরা নগরী কে প্রতিষ্ঠা করেন?
   Expected: খলিফা তৃতীয় আব্দর রহমান
   Chapter: তৃতীয় অধ্যায়

5. কর্ডোভায় সাধারণের জন্য কতটি বিরাট লাইব্রেরি ছিল?
   Expected: ১৭টি
   Chapter: চতুর্থ অধ্যায়

6. খলিফা হাকেম কতটি মূল্যবান গ্রন্থ সংগ্রহ করেছিলেন?
   Expected: প্রায় ছয় লক্ষ
   Chapter: চতুর্থ অধ্যায়

7. আলহামরা প্রাসাদের ভিত্তি কে স্থাপন করেন?
   Expected: সোলতান ইবনে আলআহমর
   Chapter: অষ্টম অধ্যায়

8. গ্রাণাডায় কোন কোন শিল্পের উন্নতি হয়েছিল?
   Expected: কাগজ, রেশমী বস্ত্র, অস্ত্রশস্ত্র, খোদাই ও অন্যান্য শিল্প
   Chapter: দশম অধ্যায়

9. গ্রাণাডায় কতটি কলেজ ও উচ্চশ্রেণীর 

In [14]:
import time

test_results = []

for i, test in enumerate(test_cases, 1):
    question = test["question"]
    expected = test["expected"]

    try:
        actual = chatbot_reply(question, [])

        test_results.append({
            "question": question,
            "expected": expected,
            "actual": actual
        })

        print(f"\n--- Test {i} ---")
        print("Question:", question)
        print("Expected:", expected)
        print("Actual:", actual)
        print("-" * 50)

    except Exception as e:
        print(f"Test {i} failed:", e)
        break

    time.sleep(2)

print("\nTests completed:", len(test_results), "/ 10")


--- Test 1 ---
Question: কর্ডোভা নগরী কেন বিখ্যাত ছিল?
Expected: সভ্যতা, শিক্ষা, সৌন্দর্য, শিল্প ও বাণিজ্যের জন্য
Actual: কর্ডোভা মহানগরী প্রাচীনকালে সভ্যতা, সৌন্দর্য ও শিক্ষার বিচিত্র লীলাভূমি ও কীর্ত্তিমন্দির বলিয়া খ্যাতিলাভ করিয়াছিল [1]। গৌরবের দিনে কর্ডোভার ঐশ্বর্য্য ও সৌন্দর্য্য, শিক্ষা ও সভ্যতা, শিল্প ও বাণিজ্য, সুখ ও স্বাচ্ছন্দ্য, আমোদ প্রমোদ ও বিলাস-উল্লাস এটিকে কবি-চিত্ত-সম্মোহন কল্পনাতীত সুন্দরী ও সুখময়ী করিয়া তুলিয়াছিল [1]। এছাড়া কর্ডোভা সমগ্র ইউরোপের জ্ঞান, বিদ্যা ও সভ্যতার কেন্দ্ররূপে পরিণত হইয়াছিল এবং জ্ঞানালোচনা ও শিক্ষার কোলাহলে মুখরিত এবং আবিষ্কার ও উদ্ভাবনার মহিমায় শীর্ষস্থানীয় ছিল [2]। [3]

**সূত্র:**
- [প্রথম অধ্যায়](https://bn.wikisource.org/wiki/%E0%A6%B8%E0%A7%8D%E0%A6%AA%E0%A7%87%E0%A6%A8%E0%A7%80%E0%A6%AF%E0%A6%BC_%E0%A6%AE%E0%A7%81%E0%A6%B8%E0%A6%B2%E0%A6%AE%E0%A6%BE%E0%A6%A8_%E0%A6%B8%E0%A6%AD%E0%A7%8D%E0%A6%AF%E0%A6%A4%E0%A6%BE/%E0%A6%AA%E0%A7%8D%E0%A6%B0%E0%A6%A5%E0%A6%AE_%E0%A6%85%E0%A6%A7%E0%A7%8D%E0%A6%AF%E0%A6%BE%E0%A6%AF%E0%A6%BC)
- [চতুর্থ 

In [15]:
# Launch the RAG chatbot after testing
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb66dd2938068d6ba3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cb66dd2938068d6ba3.gradio.live
